In [1]:
#---testing---
import spacy

# Load English model
nlp = spacy.load("en_core_web_sm")

# Test a simple sentence
doc = nlp("The food is tasty but service is slow.")
tokens = [token.text for token in doc]
print(tokens)

C:\Users\EUGENE\.conda\envs\smartcafeteria\lib\site-packages\requests\__init__.py:86: RequestsDependencyWarning: Unable to find acceptable character detection dependency (chardet or charset_normalizer).
  warnings.warn(


['The', 'food', 'is', 'tasty', 'but', 'service', 'is', 'slow', '.']


In [2]:
#---testing---
tokens = [token.lower() for token in tokens if token.isalpha()]
print(tokens)
# Output: ['food', 'is', 'tasty', 'but', 'service', 'is', 'slow']

['the', 'food', 'is', 'tasty', 'but', 'service', 'is', 'slow']


In [16]:
#---testing---
# === Updated Python Prototype for Knowledge-Driven FIS ===

import pandas as pd
import numpy as np
import skfuzzy as fuzz
from skfuzzy import control as ctrl
import spacy

# --- Step 1: Load spaCy ---
nlp = spacy.load("en_core_web_sm")

# --- Step 2: Load Excel keyword lexicon ---
# Excel columns: Variable | KeyWord | mu_good | mu_average | mu_poor
lexicon = pd.read_excel("Variable and keywords.xlsx")
lexicon.columns = lexicon.columns.str.strip()  # remove extra spaces

# --- Step 3: Define Fuzzy Variables ---
food = ctrl.Antecedent(np.arange(0, 1.01, 0.01), 'food')
cleanliness = ctrl.Antecedent(np.arange(0, 1.01, 0.01), 'cleanliness')
menu = ctrl.Antecedent(np.arange(0, 1.01, 0.01), 'menu')
waiting = ctrl.Antecedent(np.arange(0, 1.01, 0.01), 'waiting')
rating = ctrl.Consequent(np.arange(0, 5.1, 0.1), 'rating')

# --- Step 4: Define Fuzzy Membership Functions (triangular) ---
for var in [food, cleanliness, menu, waiting]:
    var['low'] = fuzz.trimf(var.universe, [0.0, 0.0, 0.4])
    var['medium'] = fuzz.trimf(var.universe, [0.3, 0.5, 0.7])
    var['high'] = fuzz.trimf(var.universe, [0.6, 1.0, 1.0])

rating['low'] = fuzz.trimf(rating.universe, [0.0, 0.0, 2.0])
rating['medium'] = fuzz.trimf(rating.universe, [1.0, 2.5, 4.0])
rating['high'] = fuzz.trimf(rating.universe, [3.0, 5.0, 5.0])

# --- Step 5: FIS Rules (10 prototype rules) ---
rule1 = ctrl.Rule(food['high'] & cleanliness['high'] & menu['high'] & waiting['low'], rating['high'])
rule2 = ctrl.Rule(food['high'] & waiting['high'], rating['medium'])
rule3 = ctrl.Rule(food['medium'] & cleanliness['medium'] & menu['medium'] & waiting['medium'], rating['medium'])
rule4 = ctrl.Rule(food['low'] | cleanliness['low'], rating['low'])
rule5 = ctrl.Rule(menu['low'] & waiting['high'], rating['medium'])
rule6 = ctrl.Rule(food['high'] & waiting['high'], rating['medium'])
rule7 = ctrl.Rule(cleanliness['high'] & menu['low'] & waiting['low'], rating['medium'])
rule8 = ctrl.Rule(food['medium'] & cleanliness['high'] & menu['medium'] & waiting['medium'], rating['medium'])
rule9 = ctrl.Rule(food['low'] & waiting['high'], rating['low'])
rule10 = ctrl.Rule(food['high'] & cleanliness['medium'] & menu['high'] & waiting['high'], rating['medium'])

rating_ctrl = ctrl.ControlSystem([rule1, rule2, rule3, rule4, rule5, rule6, rule7, rule8, rule9, rule10])
rating_sim = ctrl.ControlSystemSimulation(rating_ctrl)

# --- Step 6: Function to map keywords to fuzzy membership ---
def get_variable_membership(var_name, tokens):
    df = lexicon[lexicon['Variable'].str.strip() == var_name]
    df['Keyword'] = df['Keyword'].str.strip().str.lower()
    matches = df[df['Keyword'].isin([t.lower() for t in tokens])]
    if matches.empty:
        return 0.5  # default neutral
    return matches['mu_good'].mean()

# --- Step 7: Sample review ---
review_text = "The food is tasty, tables are clean, menu is okay, but the queue is long."

# --- Step 8: Extract keywords with spaCy ---
doc = nlp(review_text)
tokens = [token.lemma_.lower() for token in doc if token.is_alpha]

# --- Step 9: Map keywords to fuzzy memberships ---
food_input = get_variable_membership('Food Quality', tokens)
cleanliness_input = get_variable_membership('Cleanliness', tokens)
menu_input = get_variable_membership('Menu Variety', tokens)
waiting_input = get_variable_membership('Waiting Time', tokens)

# --- Step 10: Feed inputs into FIS ---
rating_sim.input['food'] = food_input
rating_sim.input['cleanliness'] = cleanliness_input
rating_sim.input['menu'] = menu_input
rating_sim.input['waiting'] = waiting_input

# --- Step 11: Compute FIS output ---
rating_sim.compute()
print("FIS defuzzified rating:", rating_sim.output['rating'])

# --- Step 12: Generate simple recommendations ---
recommendations = []
if food_input < 0.6:
    recommendations.append("Improve food quality")
if cleanliness_input < 0.6:
    recommendations.append("Improve cleanliness")
if menu_input < 0.6:
    recommendations.append("Increase menu variety")
if waiting_input > 0.5:
    recommendations.append("Reduce queue/waiting time")

print("Recommendations:", recommendations)

FIS defuzzified rating: 2.5
Recommendations: ['Improve food quality', 'Improve cleanliness', 'Increase menu variety']


C:\Users\EUGENE\AppData\Local\Temp\ipykernel_38460\1286646070.py:53: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['Keyword'] = df['Keyword'].str.strip().str.lower()


In [15]:
# --- Hybrid System Sample---

import pandas as pd
import numpy as np
import spacy
import skfuzzy as fuzz
from skfuzzy import control as ctrl

# --- Step 1: Load spaCy for NLP ---
nlp = spacy.load("en_core_web_sm")

# --- Step 2: Load refined keywords (with levels) ---
keywords_file = "Variable and keywords.xlsx"
lexicon = pd.read_excel(keywords_file, sheet_name='General')
lexicon.columns = lexicon.columns.str.strip()

# --- Step 3: Define fuzzy variables ---
food = ctrl.Antecedent(np.arange(0, 1.01, 0.01), 'food')
cleanliness = ctrl.Antecedent(np.arange(0, 1.01, 0.01), 'cleanliness')
menu = ctrl.Antecedent(np.arange(0, 1.01, 0.01), 'menu')
waiting = ctrl.Antecedent(np.arange(0, 1.01, 0.01), 'waiting')
rating = ctrl.Consequent(np.arange(0, 5.1, 0.1), 'rating')

for var in [food, cleanliness, menu, waiting]:
    var['low'] = fuzz.trimf(var.universe, [0, 0, 0.4])
    var['medium'] = fuzz.trimf(var.universe, [0.3, 0.5, 0.7])
    var['high'] = fuzz.trimf(var.universe, [0.6, 1, 1])

rating['low'] = fuzz.trimf(rating.universe, [0, 0, 2])
rating['medium'] = fuzz.trimf(rating.universe, [1, 2.5, 4])
rating['high'] = fuzz.trimf(rating.universe, [3, 5, 5])

# --- Step 4: Load refined rule-based txt and parse Markdown table ---
rule_file = "Rule based.txt"
rules_list = []
with open(rule_file, 'r') as f:
    for line in f:
        line = line.strip()
        if not line or line.startswith('| Rule') or line.startswith('| ----'):
            continue
        parts = line.split('|')
        if len(parts) < 4:
            continue
        condition_str = parts[2].strip()
        rating_str = parts[3].strip()
        rules_list.append((condition_str, rating_str))

# Function to convert parsed rule tuple to skfuzzy control.Rule
def create_rule(rule_tuple, output_var):
    conditions, rating_part = rule_tuple
    rating_level = rating_part.split('=')[-1].strip().lower()
    
    # Handle OR conditions first
    or_parts = conditions.split('OR')
    expr = None
    var_map = {'Food Quality': food, 'Cleanliness': cleanliness, 'Menu Variety': menu, 'Waiting Time': waiting}
    
    for or_c in or_parts:
        # Now handle AND inside each OR component
        and_parts = or_c.strip().split('AND')
        sub_expr = None
        for c in and_parts:
            c = c.strip()
            if not c:
                continue
            var_name = ' '.join(c.split()[:-1])
            level = c.split()[-1].lower()
            term = var_map[var_name][level]
            sub_expr = term if sub_expr is None else sub_expr & term
        expr = sub_expr if expr is None else expr | sub_expr  # OR across OR components
    
    return ctrl.Rule(expr, output_var[rating_level])

# Create FIS rules
fis_rules = [create_rule(r, rating) for r in rules_list[:10]]  # take first 10 for testing
rating_ctrl = ctrl.ControlSystem(fis_rules)
rating_sim = ctrl.ControlSystemSimulation(rating_ctrl)

# --- Step 5: Function to map tokens to fuzzy membership ---
def get_fuzzy_value(var_name, tokens):
    df = lexicon[lexicon['Variable'].str.strip() == var_name]
    value = 0.5  # default medium
    for _, row in df.iterrows():
        keyword_list = [k.strip().lower() for k in row['Keyword'].split(',')]
        for t in tokens:
            if t.lower() in keyword_list:
                if 'high' in row['Notes / Membership'].lower():
                    value = max(value, 1.0)
                elif 'medium' in row['Notes / Membership'].lower():
                    value = max(value, 0.5)
                else:
                    value = min(value, 0.0)
    return value

# --- Step 6: Process single review ---
review = "The food is tasty, menu has some variety, tables are clean but wait is long during peak."
doc = nlp(review)
tokens = [token.lemma_ for token in doc if token.is_alpha]

food_input = get_fuzzy_value('Food Quality', tokens)
clean_input = get_fuzzy_value('Cleanliness', tokens)
menu_input = get_fuzzy_value('Menu Variety', tokens)
wait_input = get_fuzzy_value('Waiting Time', tokens)

# --- Step 7: FIS simulation ---
rating_sim.input['food'] = food_input
rating_sim.input['cleanliness'] = clean_input
rating_sim.input['menu'] = menu_input
rating_sim.input['waiting'] = wait_input
rating_sim.compute()
fis_rating = rating_sim.output['rating']

# --- Step 8: Weighted average component ---
weights = {'food':0.4, 'menu':0.2, 'waiting':0.2, 'cleanliness':0.2}
weighted_rating = food_input*weights['food'] + menu_input*weights['menu'] + wait_input*weights['waiting'] + clean_input*weights['cleanliness']
weighted_rating = weighted_rating * 5  # scale to 5

# --- Step 9: Hybrid rating merge ---
hybrid_rating = 0.7*fis_rating + 0.3*weighted_rating

print("FIS Rating:", fis_rating)
print("Weighted Avg Rating:", weighted_rating)
print("Hybrid Rating:", hybrid_rating)

FIS Rating: 2.5
Weighted Avg Rating: 5.0
Hybrid Rating: 3.25
